In [14]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

ROOT = Path.cwd()
OUT = ROOT / "question_e"
OUT.mkdir(exist_ok=True)


def locate(stem, suffix):
    files = sorted(
        p for folder in (ROOT, ROOT / "upload") if folder.exists()
        for p in folder.glob(f"{stem}*{suffix}")
    )
    if len(files) != 1:
        raise FileNotFoundError(f"Expected one {stem}*{suffix}; found {len(files)}.")
    return files[0]



book = locate("Problem_Set8_2026", ".xlsx")
raw = pd.read_excel(book, sheet_name="125 Size-Past return portfolios", header=None)
factors = pd.read_excel(book, sheet_name="FF factors")
families = ("1m", "12m", "60m")
labels = [
    f"{family} S{s}P{p}"
    for family in families for s in range(1, 6) for p in range(1, 6)
]

blocks, missing_codes = [], {}
for offset, family, heading in zip(
    (0, 26, 52), families,
    ("Past 1-month return", "Past 2-12-month return", "Past 13-60-month return")
):
    assert raw.iloc[7, offset] == "Size"
    assert raw.iloc[8, offset].strip() == heading
    dates = pd.to_numeric(raw[offset], errors="coerce")
    rows = dates.between(190001, 210012) & dates.mod(100).between(1, 12)
    block = raw.loc[rows, offset + 1:offset + 25].apply(pd.to_numeric, errors="raise")
    missing_codes[family] = {
        str(v): int(np.isclose(block, v).sum()) for v in (-999, -99.99)
    }
    block = block.mask(np.isclose(block, -999) | np.isclose(block, -99.99)) / 100
    block.index = pd.to_datetime(
        dates[rows].astype(int).astype(str), format="%Y%m"
    ).dt.to_period("M")
    block.columns = [
        f"{family} S{s}P{p}" for s in range(1, 6) for p in range(1, 6)
    ]
    blocks.append(block)

assert all(b.index.equals(blocks[0].index) for b in blocks)
R = pd.concat(blocks, axis=1)
assert R.index.is_unique and R.index.is_monotonic_increasing
assert R.index.equals(pd.period_range(R.index[0], R.index[-1], freq="M"))


observed = R.to_numpy(dtype=float)
observed = observed[~np.isnan(observed)]
assert np.isfinite(observed).all() and (observed > -1).all(), (
    "An observed return is infinite or <= -100%; inspect units and missing codes."
)



rf = pd.Series(
    pd.to_numeric(factors["RF"], errors="raise").to_numpy(),
    index=pd.to_datetime(
        factors.iloc[:, 0].astype(int).astype(str), format="%Y%m"
    ).dt.to_period("M"),
    name="RF"
)
rf = rf.mask(np.isclose(rf, -999) | np.isclose(rf, -99.99)) / 100
assert rf.index.is_unique
rf = rf.reindex(R.index)
excess = R.sub(rf, axis=0)


seasonal_supplied = pd.read_csv(
    locate("table_partA_seasonal_all75", ".csv")
).set_index("portfolio")
fmb_supplied = pd.read_csv(
    locate("PS8_Qd_Fama_MacBeth_summary", ".csv")
).set_index("Variable")
monthly_supplied = pd.read_csv(locate("PS8_Qd_monthly_coefficients", ".csv"))
monthly_supplied.index = pd.PeriodIndex(monthly_supplied.pop("Month"), freq="M")
assert set(seasonal_supplied.index) == set(labels)
assert monthly_supplied.index.is_unique
assert monthly_supplied.index.isin(R.index).all()
assert np.allclose(
    monthly_supplied[fmb_supplied.index].mean(),
    fmb_supplied["Average Coefficient"], rtol=1e-6
)

names = ["Intercept", "Past_1M", "Past_t12_to_t2", "Past_t60_to_t13"]
log_returns = np.log1p(R)
predictors = [
    R.shift(1),
    np.expm1(log_returns.shift(2).rolling(11, min_periods=11).sum()),
    np.expm1(log_returns.shift(13).rolling(48, min_periods=48).sum())
]
X = np.stack([p.to_numpy() for p in predictors], axis=2)


eligible = np.isfinite(X).all(axis=(1, 2)) & rf.notna().to_numpy()
dates = R.index[eligible]
assert dates.equals(pd.period_range(dates[0], dates[-1], freq="M"))
R, rf, excess, X = R.loc[dates], rf.loc[dates], excess.loc[dates], X[eligible]
assert R.notna().all().all(), "An eligible realized return is missing."

cut = int(np.floor(0.70 * len(dates)))
train, test = dates[:cut], dates[cut:]
WARMUP, REFIT, HAC = 120, 12, 6
assert cut > WARMUP + REFIT and len(test) > 12

fmb = pd.DataFrame(index=train, columns=names, dtype=float)
for i, month in enumerate(train):
    design = sm.add_constant(X[i], has_constant="add")
    assert np.linalg.matrix_rank(design) == 4
    fmb.loc[month] = sm.OLS(excess.loc[month].to_numpy(), design).fit().params

jan_group = np.array([
    label.split()[1] in ("S1P1", "S1P2") for label in labels
])
dec_group = np.array([
    label.startswith("12m ") and label.endswith(("P4", "P5")) for label in labels
])


def estimate(n):
    history = train[:n]
    means = fmb.iloc[:n].mean()
    conventional_t = means / (fmb.iloc[:n].std(ddof=1) / np.sqrt(n))
    hac_t = pd.Series({
        name: sm.OLS(fmb[name].iloc[:n], np.ones((n, 1))).fit(
            cov_type="HAC", cov_kwds={"maxlags": HAC}
        ).tvalues.iloc[0]
        for name in names
    })
    beta = means.iloc[1:].where(
        (means.iloc[1:] > 0) & (conventional_t.iloc[1:] > 1.96), 0
    )

    # Seasonality NET of momentum: subtract the fitted continuation component
    # (gated training slopes x lagged-return predictors) before estimating the
    # January/December increments, so momentum is not counted twice.
    # Raw (unadjusted) estimates are kept for comparison only.
    fitted = X[:n] @ beta.to_numpy()                  # shape (n, 75)
    D = pd.DataFrame({
        "Intercept": 1.,
        "January": (history.month == 1).astype(float),
        "December": (history.month == 12).astype(float)
    }, index=history)
    records = []
    for j, label in enumerate(labels):
        raw_y = excess.loc[history, label]
        net_y = raw_y - fitted[:, j]
        net = sm.OLS(net_y, D).fit(cov_type="HC1")
        raw = sm.OLS(raw_y, D).fit(cov_type="HC1")
        records.append([
            *net.params, *net.tvalues,
            raw.params["January"], raw.tvalues["January"],
            raw.params["December"], raw.tvalues["December"]
        ])
    seasonal = pd.DataFrame(records, index=labels, columns=[
        "Intercept", "January", "December",
        "t_Intercept", "t_January", "t_December",
        "raw_January", "raw_t_January", "raw_December", "raw_t_December"
    ])
    # "January"/"December" are now the momentum-adjusted increments (used below).
    jan = seasonal["January"].where(
        jan_group & (seasonal["January"] > 0) & (seasonal["t_January"] > 1.96), 0
    ).to_numpy()
    dec = seasonal["December"].where(
        dec_group & (seasonal["December"] > 0) & (seasonal["t_December"] > 1.96), 0
    ).to_numpy()
    summary = pd.DataFrame({
        "Coefficient": means,
        "t_conventional": conventional_t,
        "t_HAC6": hac_t
    })
    return beta.to_numpy(), jan, dec, summary, seasonal


def zscore(v):
    sd = np.std(v)
    return (v - np.mean(v)) / sd if sd > 1e-12 else np.zeros_like(v)



In [15]:

wf_momentum = np.zeros((cut - WARMUP, 75))
wf_seasonal = np.zeros_like(wf_momentum)
for i in range(WARMUP, cut):
    if (i - WARMUP) % REFIT == 0:
        beta, jan, dec, _, _ = estimate(i) 
    wf_momentum[i - WARMUP] = zscore(X[i] @ beta)
    wf_seasonal[i - WARMUP] = zscore(
        jan if train[i].month == 1 else
        dec if train[i].month == 12 else np.zeros(75)
    )


def positions(momentum, seasonal, blend, k):
    score = momentum + blend * seasonal
    weights = np.zeros_like(score)
    for i, row in enumerate(score):
        if np.ptp(row) > 1e-12:
            order = np.argsort(row, kind="stable")
            weights[i, order[-k:]], weights[i, order[:k]] = 1 / k, -1 / k
    assert np.allclose(weights.sum(axis=1), 0)
    return weights


def backtest(weights, returns, cash, cost_rate=0.001, borrow_annual=0.005):
    pretrade, records = np.zeros(75), []
    for w, r, f in zip(weights, returns.to_numpy(), cash.to_numpy()):
        turnover = np.abs(w - pretrade).sum()
        gross = w @ r  # Equals w @ excess because net risky exposure is zero.
        cost = cost_rate * turnover
        borrow = (borrow_annual / 12) * (-w[w < 0].sum())
        net = gross - cost - borrow
        total = f + net
        assert 1 + total > 0, "Strategy equity exhausted."
        records.append((gross, net, total, turnover, cost, borrow))
        pretrade = w * (1 + r) / (1 + total)
    return pd.DataFrame(records, index=returns.index, columns=[
        "Gross_Excess", "Net_Excess", "Net_Total",
        "Turnover", "Trading_Cost", "Borrow_Cost"
    ])


selection = []
for blend in (0.5, 1.0, 2.0):
    for fraction in (0.10, 0.20):
        k = int(np.floor(75 * fraction))
        w = positions(wf_momentum, wf_seasonal, blend, k)
        result = backtest(w, R.loc[train[WARMUP:]], rf.loc[train[WARMUP:]])
        selection.append({
            "seasonal_blend": blend, "k_per_leg": k,
            "training_net_SR": result.Net_Excess.mean() / result.Net_Excess.std(ddof=1)
        })
selection = pd.DataFrame(selection).sort_values(
    ["training_net_SR", "seasonal_blend", "k_per_leg"],
    ascending=[False, True, True]
)
assert np.isfinite(selection.training_net_SR).all()
best = selection.iloc[0]
beta, jan, dec, fmb_summary, seasonal = estimate(cut)

frozen = {
    "source_workbook": book.name,
    "portfolio_order": labels,
    "input_units": "simple monthly percent; divide by 100",
    "missing_codes": missing_codes,
    "eligibility": "all 75 complete lagged predictor vectors; contiguous monthly sample",
    "training_start": str(train[0]), "training_end": str(train[-1]),
    "testing_start": str(test[0]), "testing_end": str(test[-1]),
    "training_fraction": 0.70,
    "predictor_lags": {
        "Past_1M": [1, 1], "Past_t12_to_t2": [2, 12], "Past_t60_to_t13": [13, 60]
    },
    "compounding": "product(1 + raw decimal returns) - 1",
    "fmb_beta": dict(zip(names[1:], beta.tolist())),
    "beta_gate": "positive mean slope and conventional Fama-MacBeth t > 1.96",
    "seasonal_january": dict(zip(labels, jan.tolist())),
    "seasonal_december": dict(zip(labels, dec.tolist())),
    "seasonal_gate": (
        "positive joint-dummy coefficient and HC1 t > 1.96; "
        "January S1P1/S1P2 across families; December 12m P4/P5"
    ),
    "score": (
        "cross-sectional z(FMB slope forecast) + seasonal_blend * "
        "cross-sectional z(active seasonal coefficient); zero variance => zero component"
    ),
    "seasonal_blend": float(best.seasonal_blend),
    "k_per_leg": int(best.k_per_leg),
    "positions": (
        "top k equally weighted +1; bottom k equally weighted -1; "
        "all-zero scores => cash; shorts are relative hedges"
    ),
    "ties": "stable ascending sort in portfolio_order; last k long, first k short",
    "rebalancing": "monthly at prior month-end; coefficients fixed throughout testing",
    "gross_exposure": 2.0, "net_exposure": 0.0,
    "collateral": "100% equity earns RF; no additional funding spread",
    "cost_per_dollar_traded": 0.001,
    "short_borrow_annual": 0.005,
    "cost_accounting": (
        "month-end deductions; trading cost times absolute target-minus-drifted weights; "
        "borrow = annual rate / 12 times short exposure"
    ),
    "initialization": "each evaluation segment starts flat; no terminal liquidation",
    "selection": "highest net walk-forward training Sharpe; ties favor lower blend then k",
    "warmup_months": WARMUP, "training_refit_interval": REFIT,
    "candidate_blends": [0.5, 1.0, 2.0], "candidate_k": [7, 15],
    "sharpe": "excess mean / sample SD; annualized = sqrt(12) * monthly",
    "caveat": "Step 1 screened full-sample information; this is not an independent discovery test."
}
import hashlib
from datetime import datetime, timezone

# ---- FREEZE (no silent overwrite) -------------------------------------------
# v1 = the original notebook, whose test results were already viewed.
# This momentum-adjusted rule is therefore v2 and must be disclosed as a
# post-hoc revision. Change FREEZE_VERSION only when you deliberately start
# a new version.
FREEZE_VERSION = 2

frozen.update({
    "freeze_version": FREEZE_VERSION,
    "supersedes": (
        "PS8_Qe_frozen_strategy.json (v1; raw seasonal coefficients)"
        if (OUT / "PS8_Qe_frozen_strategy.json").exists() else None
    ),
    "seasonal_definition": (
        "January/December increments estimated on excess return MINUS the fitted "
        "momentum component (gated FMB slopes x lagged predictors), joint dummy "
        "model with Feb-Nov baseline; momentum is not counted twice"
    ),
    "seasonal_gate": (
        "positive momentum-adjusted coefficient and HC1 t > 1.96; "
        "January S1P1/S1P2 across families; December 12m P4/P5"
    ),
    "score": (
        "cross-sectional z(FMB slope forecast) + seasonal_blend * "
        "cross-sectional z(active momentum-adjusted seasonal coefficient); "
        "zero variance => zero component"
    ),
    "revision_note": (
        "Rule revised after the v1 test-period results had been viewed; "
        "the test period is therefore not a clean pre-registered holdout for v2."
        if FREEZE_VERSION > 1 else "Original freeze."
    ),
    "caveat": (
        "Step 1 screened full-sample (1936-2026) information, including the test "
        "period, to nominate the seasonal groups and signal families. Coefficients "
        "are training-only, but the test period is not an independent discovery test."
    ),
})


def same_rule(a, b):
    """Structural equality with a tiny numeric tolerance (library/platform noise)."""
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(same_rule(a[k], b[k]) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(same_rule(x, y) for x, y in zip(a, b))
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return bool(np.isclose(a, b, rtol=1e-9, atol=1e-12))
    return a == b


rule_path = OUT / f"PS8_Qe_frozen_strategy_v{FREEZE_VERSION}.json"
payload = json.loads(json.dumps(frozen, allow_nan=False))
if rule_path.exists():
    if not same_rule(json.loads(rule_path.read_text()), payload):
        raise RuntimeError(
            f"{rule_path.name} already exists and differs from the rule just fitted. "
            "Refusing to overwrite a frozen strategy. If the change is intentional, "
            "increase FREEZE_VERSION and disclose it as a new version."
        )
    print(f"Existing {rule_path.name} matches the fitted rule; keeping it unchanged.")
else:
    text = json.dumps(payload, indent=2, allow_nan=False)
    with rule_path.open("x", encoding="utf-8") as handle:   # 'x' fails if file exists
        handle.write(text)
    (OUT / (rule_path.name + ".sha256")).write_text(
        f"{hashlib.sha256(text.encode()).hexdigest()}  {rule_path.name}  "
        f"frozen_at_utc={datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
    )
    print(f"Froze {rule_path.name}.")
frozen = json.loads(rule_path.read_text())

Existing PS8_Qe_frozen_strategy_v2.json matches the fitted rule; keeping it unchanged.


In [16]:
S_WF = "Train walk-forward (re-estimated every 12m)"
S_FR = "Train final-frozen rule (in-sample, not unbiased)"
S_TE = "Test frozen (fixed at end of training)"
COSTS = (frozen["cost_per_dollar_traded"], frozen["short_borrow_annual"])

train_w = positions(
    wf_momentum, wf_seasonal, frozen["seasonal_blend"], frozen["k_per_leg"]
)
train_result = backtest(
    train_w, R.loc[train[WARMUP:]], rf.loc[train[WARMUP:]], *COSTS
)
beta = np.array([frozen["fmb_beta"][name] for name in names[1:]])
jan = np.array([frozen["seasonal_january"][label] for label in labels])
dec = np.array([frozen["seasonal_december"][label] for label in labels])


def frozen_scores(X_block, months):
    momentum = np.array([zscore(x @ beta) for x in X_block])
    seasonal_part = np.array([
        zscore(jan if m.month == 1 else dec if m.month == 12 else np.zeros(75))
        for m in months
    ])
    return momentum, seasonal_part


train_months = train[WARMUP:]
fm_, fs_ = frozen_scores(X[WARMUP:cut], train_months)
train_frozen_w = positions(fm_, fs_, frozen["seasonal_blend"], frozen["k_per_leg"])
train_frozen_result = backtest(
    train_frozen_w, R.loc[train_months], rf.loc[train_months], *COSTS
)

tm_, ts_ = frozen_scores(X[cut:], test)
test_w = positions(tm_, ts_, frozen["seasonal_blend"], frozen["k_per_leg"])
test_result = backtest(test_w, R.loc[test], rf.loc[test], *COSTS)

rows = []
for sample, coefficients, result in [
    (S_WF, "re-estimated every 12 months (prior data only)", train_result),
    (S_FR, "final training coefficients, fixed", train_frozen_result),
    (S_TE, "final training coefficients, fixed", test_result),
]:
    for column in ("Gross_Excess", "Net_Excess"):
        v = result[column]
        mean, sd = v.mean(), v.std(ddof=1)
        t = sm.OLS(v.to_numpy(), np.ones((len(v), 1))).fit(
            cov_type="HAC", cov_kwds={"maxlags": HAC}
        ).tvalues[0]
        rows.append([
            sample, column, coefficients, str(v.index[0]), str(v.index[-1]), len(v),
            mean, sd, mean / sd, np.sqrt(12) * mean / sd, t
        ])
performance = pd.DataFrame(rows, columns=[
    "Sample", "Returns", "Coefficients", "Start", "End", "N", "Mean_monthly",
    "SD_monthly", "SR_monthly", "SR_annualized", "Mean_t_HAC6"
]).set_index(["Sample", "Returns"])

monthly = pd.concat([
    train_result.assign(Sample="Training walk-forward"),
    test_result.assign(Sample="Testing")
])
monthly["RF"] = rf.reindex(monthly.index)
monthly.index.name = "Month"
weights = pd.DataFrame(
    np.vstack([train_w, test_w]), index=monthly.index, columns=labels
)
assert np.allclose(
    (weights * excess.loc[monthly.index]).sum(axis=1), monthly.Gross_Excess
)
assert np.allclose(train_frozen_w.sum(axis=1), 0)

split = pd.Series("Training calibration", index=dates, name="Sample")
split.loc[train[WARMUP:]], split.loc[test] = "Training walk-forward", "Testing"
split.to_csv(OUT / "PS8_Qe_sample_split.csv", index_label="Month")
monthly.to_csv(OUT / "PS8_Qe_monthly_returns.csv")
weights.to_csv(OUT / "PS8_Qe_monthly_weights.csv")
train_frozen_result.assign(Sample="Training final-frozen (in-sample)").to_csv(
    OUT / "PS8_Qe_train_final_frozen_rule_returns.csv", index_label="Month"
)
performance.to_csv(OUT / "PS8_Qe_performance.csv")
selection.to_csv(OUT / "PS8_Qe_training_candidates.csv", index=False)
fmb.to_csv(OUT / "PS8_Qe_training_monthly_FMB.csv", index_label="Month")
fmb_summary.to_csv(OUT / "PS8_Qe_training_FMB_summary.csv", index_label="Variable")
seasonal.to_csv(OUT / "PS8_Qe_training_seasonal.csv", index_label="portfolio")
excess.assign(RF=rf, Sample=split).to_csv(
    OUT / "PS8_Qe_portfolio_excess_for_Qg.csv", index_label="Month"
)

wf_sr = performance.loc[(S_WF, "Net_Excess"), "SR_annualized"]
fr_sr = performance.loc[(S_FR, "Net_Excess"), "SR_annualized"]
test_sr = performance.loc[(S_TE, "Net_Excess"), "SR_annualized"]
test_t = performance.loc[(S_TE, "Net_Excess"), "Mean_t_HAC6"]
verdict = (
    "is positive but weaker" if 0 < test_sr < wf_sr else
    "is positive" if test_sr > 0 else "does not remain positive"
)
assessment = (
    f"Net annualized Sharpe: training walk-forward {wf_sr:.3f}; final frozen rule "
    f"over the training months {fr_sr:.3f} (in-sample, not an unbiased backtest); "
    f"test {test_sr:.3f}; test performance {verdict} (HAC mean t={test_t:.2f}). "
    "The training walk-forward re-estimates coefficients every "
    f"{REFIT} months using prior data only, whereas the test uses coefficients fixed "
    "at the end of training; the two are not like-for-like. "
    f"Split: {cut} training / {len(test)} testing months "
    f"({test[0]} to {test[-1]}); the first {WARMUP} training months calibrate the model "
    "and are excluded from performance. "
    "Seasonal increments are estimated net of the fitted momentum component. "
    "Parameter selection optimizes the walk-forward training Sharpe. "
    "Step 1 screened full-sample information (including the test period), so the "
    "test period is not an independent discovery test"
    + ("; this rule is version "
       f"{frozen['freeze_version']}, revised after earlier test results were viewed. "
       if frozen["freeze_version"] > 1 else ". ")
    + "Annualization uses sqrt(12) without autocorrelation adjustment. "
    "Costs are assumptions and overlapping portfolios retain factor risks. "
    "Use Net_Excess for (f) and match sample dates and excess-return units in (g)."
)
(OUT / "PS8_Qe_assessment.txt").write_text(assessment)

shown = performance.copy()
shown[["Mean_monthly", "SD_monthly"]] *= 100
shown = shown.rename(columns={
    "Mean_monthly": "Mean_monthly_%", "SD_monthly": "SD_monthly_%"
})
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(shown.round(3))
print(assessment)

Coefficients    Start      End    N  Mean_monthly_%  SD_monthly_%  SR_monthly  SR_annualized  \
Sample                                            Returns                                                                                                                                        
Train walk-forward (re-estimated every 12m)       Gross_Excess  re-estimated every 12 months (prior data only)  1953-03  2001-06  580           1.434         3.237       0.443          1.534   
                                                  Net_Excess    re-estimated every 12 months (prior data only)  1953-03  2001-06  580           1.243         3.221       0.386          1.336   
Train final-frozen rule (in-sample, not unbiased) Gross_Excess              final training coefficients, fixed  1953-03  2001-06  580           1.474         3.271       0.450          1.560   
                                                  Net_Excess                final training coefficients, fixed  1953-03  2001-06  580           1.272         3.260       0.390          1.352   
Test frozen (fixed at end of training)            Gross_Excess              final training coefficients, fixed  2001-07  2026-06  300           0.382         3.837       0.100          0.345   
                                                  Net_Excess                final training coefficients, fixed  2001-07  2026-06  300           0.120         3.835       0.031          0.108   

                                                                Mean_t_HAC6  
Sample                                            Returns                    
Train walk-forward (re-estimated every 12m)       Gross_Excess       10.714  
                                                  Net_Excess          9.330  
Train final-frozen rule (in-sample, not unbiased) Gross_Excess       10.940  
                                                  Net_Excess          9.446  
Test frozen (fixed at end of training)            Gross_Excess        1.950  
                                                  Net_Excess          0.607

Net annualized Sharpe: training walk-forward 1.336; final frozen rule over the training months 1.352 (in-sample, not an unbiased backtest); test 0.108; test performance is positive but weaker (HAC mean t=0.61). The training walk-forward re-estimates coefficients every 12 months using prior data only, whereas the test uses coefficients fixed at the end of training; the two are not like-for-like. Split: 700 training / 300 testing months (2001-07 to 2026-06); the first 120 training months calibrate the model and are excluded from performance. Seasonal increments are estimated net of the fitted momentum component. Parameter selection optimizes the walk-forward training Sharpe. Step 1 screened full-sample information (including the test period), so the test period is not an independent discovery test; this rule is version 2, revised after earlier test results were viewed. Annualization uses sqrt(12) without autocorrelation adjustment. Costs are assumptions and overlapping portfolios retain 